# Digital Forensics Investigation: "The Stolen Szechuan Sauce"

This notebook demonstrates an end-to-end digital forensics investigation using the **Sec-Gemini Python SDK** and **Bring-Your-Own-Tool (BYOT)** architecture to solve the famous DFIR challenge **"The Stolen Szechuan Sauce"**.

### 🔍 The Challenge Scenario
* **Case Name:** The Stolen Szechuan Sauce (Case 001)
* **Original Author & Source:** [DFIR Madness](https://dfirmadness.com/the-stolen-szechuan-sauce/)
* **Incident Summary:** A top-secret, highly valuable recipe for Szechuan sauce has been stolen from a corporate domain controller (`DC01`). The organization suspects an advanced breach or insider compromise.
* **Investigation Objective:** As a digital forensics investigator, your mission is to analyze the server's logs (`DC01-E01`), detect and reconstruct the attacker's activities.

---

### 🛠️ Workflow Overview
1. **Setup & Ingestion:** Download `sauce.db`, a pre-built SQLite logs database containing the complete forensic timeline of the compromised domain controller. *(For instructions on building this database from raw E01 disk images using Plaso, see the **Appendix** at the bottom).*
2. **Tool Tunneling:** Serve the logs database via a local FastMCP server and use BYOT to establish a secure tunnel to the Sec-Gemini cloud agent.
3. **AI Investigation:** Initialize the Python SDK, prompt the agent in `dfir` mode, and stream the autonomous forensic analysis in real time.

## 1. Dependencies

Install the Sec-Gemini SDK, FastMCP, and gdown.

In [ ]:
!pip install -q sec-gemini fastmcp gdown

## 2. Authentication

Set your Sec-Gemini API key. In Google Colab, add a secret named `SEC_GEMINI_API_KEY` in the Secrets tab (key icon on the left) and enable notebook access.

In [ ]:
import os

API_KEY = os.environ.get("SEC_GEMINI_API_KEY")

if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        API_KEY = userdata.get("SEC_GEMINI_API_KEY")
    except (ImportError, Exception):
        pass

if not API_KEY:
    API_KEY = "YOUR_API_KEY_HERE"

assert API_KEY and API_KEY != "YOUR_API_KEY_HERE", "Please set your API key"

## 3. Ingest Sample Logs Database (`sauce.db`)

`sauce.db` is a pre-built SQLite database containing forensic log records and descriptions which is generated from the official DFIR Madness domain controller disk image (`DC01-E01.zip`). 

You can check the **Appendix: Building sauce.db from Scratch** at the bottom of this notebook to see the exact Plaso CLI commands and Python parsing scripts used to build `sauce.db`.

In [ ]:
import os

# Download the compressed sample database (sauce.db.gz)
print("Downloading compressed sample database...")
!gdown "https://drive.google.com/uc?id=1c380HYIfAeKtmDNSAj978uJfIaT0WuV9" -O sauce.db.gz

print("Decompressing database...")
!gunzip -f sauce.db.gz

assert os.path.exists("sauce.db"), (
    "Failed to download or decompress sample logs database 'sauce.db'."
)
print("Sample logs database 'sauce.db' is ready.")

## 4. Start BYOT with DFIR SQLite MCP Server

We create a FastMCP server, attach the SQLite DFIR tools to it, and start the Bring-Your-Own-Tool (BYOT) service. This establishes a secure tunnel so the Sec-Gemini cloud agent can query our local `sauce.db` database during the investigation.

In [ ]:
from fastmcp import FastMCP

from sec_gemini.byot.service import ByotService
from sec_gemini.logs_mcp.backends.sqlite import mcp as sqlite_mcp

# Create FastMCP server and attach SQLite DFIR tools
dfir_mcp = FastMCP("dfir-sqlite-tools")
sqlite_mcp.make_mcp(dfir_mcp, "sauce.db")

# Start BYOT service to tunnel the DFIR tools to the cloud agent
byot = ByotService(api_key=API_KEY, name="dfir-notebook")
await byot.start(tools=[dfir_mcp])

status = byot.status()
print(f"BYOT state: {status.state}")
print(f"Tools registered: {status.tool_count}")
for tool in status.tools:
    print(f"  - {tool.name}")

## 5. Run Sec-Gemini Digital Forensics Investigation

With the BYOT tunnel active, we use the Python SDK to create a session, activate the `byot` tool tunnel, and prompt the agent in `dfir` mode.

> ⚠️ **Long-Running Investigation Warning:** A comprehensive digital forensics investigation on a domain controller dataset is an extensive autonomous analysis. The agent will repeatedly query and correlate thousands of log records. **This streaming analysis can take up to 1 hour to complete.**

### 🔄 Manual Session Resumption Guide
If your browser disconnects or your Colab session times out during this 1-hour investigation, your cloud job continues running! You can easily reconnect to it:
1. When you first run the investigation cell below, copy the printed session ID (`Session created: <YOUR_SESSION_ID>`). Store this ID in a safe place (like a notepad).
2. If you get disconnected and restart your Colab VM, simply paste your ID into `SESSION_ID = "YOUR_SESSION_ID"` in the cell below and re-run it.
3. The SDK will instantly reconnect to your running cloud job, fetch the complete past investigation history, and pick up live streaming right where it left off!

In [ ]:
from sec_gemini import SecGemini

client = SecGemini(api_key=API_KEY)
await client.start()

# --- Manual Resumption Placeholder ---
# If your Colab kernel disconnected during a long run, paste your previous session ID string here:
SESSION_ID = ""

if SESSION_ID:
    print(f"Reconnecting to existing cloud session: {SESSION_ID}")
    session = await client.sessions.get(SESSION_ID)
else:
    session = await client.sessions.create()
    print("\n============================================================")
    print(f"🎉 Session created: {session.id}")
    print("⚠️ COPY THIS ID: Store it in a safe place to resume if disconnected.")
    print("============================================================\n")
    print("Starting 1-hour forensics investigation...")
    await session.prompt(
        "Perform a forensics investigation on the given logs.",
        meta={"mode": "dfir"},
    )

# Ensure BYOT tool tunnel is active for this session
await session.mcps.set(["byot"])

print("Streaming investigation updates...")


def truncate(text: str, max_len: int = 100) -> str:
    clean_text = text.replace("\n", " ").strip()
    return clean_text[:max_len] + ("..." if len(clean_text) > max_len else "")


async for msg in session.messages.stream():
    msg_type = msg.get("message_type", "")
    content = msg.get("content", "")

    if msg_type == "MESSAGE_TYPE_RESPONSE":
        print(f"Agent: {content}")
    elif msg_type == "MESSAGE_TYPE_THOUGHT":
        print(f"  [thought] {truncate(content)}\n")
    elif msg_type == "MESSAGE_TYPE_TOOL_CALL":
        print(f"  [tool] {msg.get('title', '')} ({truncate(content)})")
    elif msg_type == "MESSAGE_TYPE_TOOL_RESULT":
        print(f"    ↳ [result] {truncate(content)}\n")

## 6. Cleanup

Stop the BYOT service, delete the session, and close the SDK client.

> ⚠️ **Irreversible Action:** Executing `session.delete()` permanently destroys the session and its investigation history from the Sec-Gemini cloud backend. **Once deleted, the session cannot be recovered.**

In [ ]:
# Permanently delete the session from the cloud backend (this cannot be undone)
await session.delete()

# Close SDK connection and stop BYOT tunnel
await client.close()
await byot.stop()
print("Cleanup complete.")

---
## Appendix: Building `sauce.db` from Scratch

This section is provided for reference. It documents the exact commands and Python helper functions used to create the `sauce.db` SQLite database from a raw server disk image (`DC01-E01.zip`) using the Plaso forensics engine (`log2timeline` / `psort`).

*(Note: Executing this section will perform a full Plaso disk image processing run, which may take significant time).*

In [ ]:
# Install Plaso using uv for parallel dependency resolution
!pip install -q uv
!uv pip install plaso

In [ ]:
# Access sample E01 disk image zip (DC01-E01.zip)
# Choose one of the two ingestion methods below:

# --- Option 1: Direct Web Download (Official DFIR Madness Server) ---
# Download the original disk image directly to your VM disk:
# %env SERVER_DISK_IMAGE_ZIP=/tmp/DC01-E01.zip
# !curl -L -o $SERVER_DISK_IMAGE_ZIP https://dfirmadness.com/case001/DC01-E01.zip

# --- Option 2: Colab Native Drive Mount (Google Drive Mirror) ---
# To access a copy of the file efficiently without consuming Colab VM disk space or encountering download delays:
# 1. Open the Drive link in your browser: https://drive.google.com/file/d/1gtjdk48Bj8KPSpQ3PDskRbJL9_V0N78-/view?usp=drive_link&resourcekey=0-550RRcMrKZtonuQwRonZCg
# 2. Click "Add shortcut to Drive" and place it in your MyDrive.
# 3. Mount your Drive in Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# %env SERVER_DISK_IMAGE_ZIP=/content/drive/MyDrive/DC01-E01.zip

!unzip $SERVER_DISK_IMAGE_ZIP -d /tmp/

In [ ]:
# Run log2timeline to generate plaso storage file
!log2timeline --partition all --vss_stores all --storage_file timeline.plaso /tmp/E01-DC01/20200918_0347_CDrive.E01

In [ ]:
# Export plaso events to JSON Lines
!psort -o json_line -w events.jsonl timeline.plaso

In [ ]:
# Python helper functions to parse events.jsonl into SQLite schema
import json
import os
import sqlite3

PLASO_DATA_TYPE_DESCRIPTIONS = {
    "android:app_usage": "Android application usage event data.",
    "android:event:battery": "Android turbo battery event data.",
    "android:event:call": "Android Call event data.",
    "android:logcat": "Android logcat event data.",
    "android:messaging:hangouts": "Google Hangouts Message event data.",
    "android:messaging:sms": "Android SMS event data.",
    "android:sqlite:app_usage": "Android app usage event data.",
    "android:tango:contact": "Tango on Android contact event data.",
    "android:tango:conversation": "Tango on Android conversation event data.",
    "android:tango:message": "Tango on Android message event data.",
    "android:twitter:contact": "Twitter on Android contact event data.",
    "android:twitter:search": "Twitter on Android search event data.",
    "android:twitter:status": "Twitter on Android status event data.",
    "android:webview:cookie": "Android WebView cookie event data.",
    "android:webviewcache": "Android WebViewCache event data.",
    "apache:access_log:entry": "Apache access log event data.",
    "av:defender:detection_history": "Windows Defender scan DetectionHistory event data.",
    "av:mcafee:accessprotectionlog": "McAfee AV Log event data.",
    "av:symantec:scanlog": "Symantec event data.",
    "av:trendmicro:scan": "Trend Micro AV Log event data.",
    "av:trendmicro:webrep": "Trend Micro Web Reputation Log event data.",
    "aws:cloudtrail:entry": "AWS CloudTrail log event data.",
    "aws:elb:access": "AWS Elastic Load Balancer access log event data.",
    "azure:activitylog:entry": "Azure activity log event data.",
    "azure:application_gateway_access:entry": "Azure application gateway access log event data.",
    "bash:history:entry": "Bash history log event data.",
    "bsm:entry": "Basic Security Module (BSM) audit event data.",
    "ccleaner:configuration": "CCleaner configuration event data.",
    "ccleaner:update": "CCleaner update event data.",
    "chrome:autofill:entry": "Chrome Autofill event data.",
    "chrome:cache:entry": "Chrome Cache event data.",
    "chrome:cookie:entry": "Chrome Cookie event data.",
    "chrome:extension_activity:activity_log": "Chrome Extension Activity event data.",
    "chrome:history:file_downloaded": "Chrome History file downloaded event data.",
    "chrome:history:page_visited": "Chrome History page visited event data.",
    "chrome:preferences:content_settings:exceptions": "Chrome content settings exceptions event data.",
    "chrome:preferences:extension_installation": "Chrome extension event data.",
    "chrome:preferences:extensions_autoupdater": "Chrome Extension Autoupdater event data.",
    "confluence:access": "Confluence access event data.",
    "cookie:google:analytics:utma": "Google analytics __utma cookie event data.",
    "cookie:google:analytics:utmb": "Google analytics __utmb cookie event data.",
    "cookie:google:analytics:utmt": "Google analytics __utmt cookie event data.",
    "cookie:google:analytics:utmz": "Google analytics __utmz cookie event data.",
    "cri:container:log:entry": "CRI log event data.",
    "cups:ipp:event": "CUPS IPP event data.",
    "docker:container:configuration": "Docker container configuration event data.",
    "docker:container:log:entry": "Docker container log event data.",
    "docker:layer:configuration": "Docker layer configuration event data.",
    "docker:json:container:log": "Docker Container Logs.",
    "docker:json:container": "Docker Container Logs.",
    "dropbox:sync_history:entry": "Dropbox Sync History Database event data.",
    "edge:resources:load_statistics": "Microsoft Edge load statistics resource event data.",
    "event": "Formatter for events that do not have any defined formatter.",
    "file_entry": "File entry event source.",
    "firefox:cache:record": "Firefox cache event data.",
    "firefox:cookie:entry": "Firefox Cookie event data.",
    "firefox:downloads:download": "Firefox download event data.",
    "firefox:places:bookmark": "Firefox bookmark event data.",
    "firefox:places:bookmark_annotation": "Firefox bookmark annotation event data.",
    "firefox:places:bookmark_folder": "Firefox bookmark folder event data.",
    "firefox:places:page_visited": "Firefox page visited event data.",
    "fish:history:entry": "Fish history log event data.",
    "fs:bodyfile:entry": "Bodyfile event data.",
    "fs:ntfs:usn_change": "NTFS USN change event data.",
    "fs:stat": "File system stat event data.",
    "fs:stat:ntfs": "NTFS file system stat event data.",
    "gcp:log:entry": "Google Cloud (GCP) log event data.",
    "gcp:log:json": "Google Cloud (GCP) json data.",
    "gdrive:snapshot:cloud_entry": "Google Drive snapshot cloud entry event data.",
    "gdrive:snapshot:local_entry": "Google Drive snapshot local entry event data.",
    "google_drive_sync_log:entry": "Google Drive Sync log event data.",
    "googlelog:log": "Google-formatted log file event data.",
    "iis:log:line": "IIS log event data.",
    "imessage:event:chat": "iMessage and SMS event data.",
    "ios:app_privacy:access": "iOS application privacy report event of type access.",
    "ios:app_privacy:network": "iOS application privacy report event of type network activity.",
    "ios:carplay:history:entry": "Apple iOS Car Play application history event data.",
    "ios:datausage:event": "iOS datausage event data.",
    "ios:idstatuscache:lookup": "iOS identity services status cache event data.",
    "ios:kik:messaging": "Kik message event data.",
    "ios:lockdownd_log:entry": "iOS lockdown daemon (lockdownd) log event data.",
    "ios:netusage:process": "iOS netusage process event data.",
    "ios:netusage:route": "iOS netusage connection event data.",
    "ios:powerlog:application_usage": "iOS powerlog file application usage event data.",
    "ios:screentime:event": "iOS Screen Time file usage event data.",
    "ios:sysdiag_log:entry": "iOS sysdiagnose log event data.",
    "ios:sysdiagnose:logd:line": "iOS sysdiagnose logd event data.",
    "ios:twitter:contact": "Twitter on iOS 8+ contact event data.",
    "ios:twitter:status": "Parent class for Twitter on iOS 8+ status events.",
    "ipod:device:entry": "iPod plist event data.",
    "java:download:idx": "Java IDX cache file event data.",
    "kodi:videos:viewing": "Kodi video event data.",
    "linux:apt_history_log:entry": "APT History log event data.",
    "linux:dpkg_log:entry": "Dpkg event data.",
    "linux:locate_database:entry": "Linux locate database (updatedb) event data.",
    "linux:popularity_contest_log:entry": "Popularity Contest event data.",
    "linux:popularity_contest_log:session": "Popularity Contest session event data.",
    "linux:utmp:event": "Linux libc6 utmp event data.",
    "mackeeper:cache": "MacKeeper Cache event data.",
    "macos:airport:entry": "MacOS airport event data.",
    "macos:appfirewall_log:entry": "MacOS Application firewall log (appfirewall.log) file event data.",
    "macos:apple_account:entry": "Apple account event data.",
    "macos:application_usage:entry": "MacOS application usage event data.",
    "macos:asl:entry": "Apple System Log (ASL) event data.",
    "macos:asl:file": "Apple System Log (ASL) file event data.",
    "macos:background_items:entry": "Mac OS background item event data.",
    "macos:bluetooth:entry": "MacOS Bluetooth event data.",
    "macos:document_versions:file": "MacOS document revision event data.",
    "macos:fseventsd:record": "MacOS file system event (fseventsd) event data.",
    "macos:install_history:entry": "MacOS install history event data.",
    "macos:keychain:application": "MacOS keychain application password record event data.",
    "macos:keychain:internet": "MacOS keychain internet record event data.",
    "macos:knowledgec:application": "KnowledgeC application execution event data.",
    "macos:knowledgec:safari": "MacOS Duet/KnowledgeC database event data for Safari.",
    "macos:launchd:entry": "MacOS launchd event data.",
    "macos:launchd_log:entry": "Mac OS launchd log event data.",
    "macos:login_items:entry": "Mac OS login item event data.",
    "macos:login_window:entry": "Mac OS login window event data.",
    "macos:login_window:managed_login_item": "Mac OS login window managed login item event data.",
    "macos:lsquarantine:entry": "MacOS launch services quarantine event data.",
    "macos:notes:entry": "MacOS Notes event data.",
    "macos:notification_center:entry": "MacOS NotificationCenter event data.",
    "macos:securityd_log:entry": "MacOS securityd log event data.",
    "macos:software_updata:entry": "MacOS software update event data.",
    "macos:startup_item:entry": "Mac OS startup item event data.",
    "macos:tcc_entry": "MacOS TCC event data.",
    "macos:time_machine:backup": "MacOS TimeMachine backup event data.",
    "macos:unified_logging:event": "Apple Unified Logging (AUL) event data.",
    "macos:user:entry": "MacOS user event data.",
    "macos:utmpx:entry": "MacOS utmpx event data.",
    "macos:wifi_log:entry": "MacOS Wi-Fi log event data.",
    "microsoft365:audit_log:entry": "Microsoft (Office) 365 audit log event data.",
    "msie:webcache:container": "MSIE WebCache Container table event data.",
    "msie:webcache:containers": "MSIE WebCache Containers table event data.",
    "msie:webcache:cookie": "MSIE WebCache Container table event data.",
    "msie:webcache:leak_file": "MSIE WebCache LeakFiles event data.",
    "msie:webcache:partitions": "MSIE WebCache Partitions table event data.",
    "msiecf:leak": "MSIECF leak event data.",
    "msiecf:redirected": "MSIECF redirected event data.",
    "msiecf:url": "MSIECF URL event data.",
    "networkminer:fileinfos:file": "NetworkMiner event Data.",
    "olecf:dest_list:entry": ".automaticDestinations-ms DestList entry event data.",
    "olecf:document_summary_info": "OLECF document summary information event data.",
    "olecf:item": "OLECF item event data.",
    "olecf:summary_info": "OLECF summary information event data.",
    "openxml:metadata": "OXML event data.",
    "opera:history:entry": "Opera global history entry data.",
    "opera:history:typed_entry": "Opera typed history entry data.",
    "p2p:bittorrent:transmission": "Transmission BitTorrent event data.",
    "p2p:bittorrent:utorrent": "UTorrent active torrent event data.",
    "pe_coff:dll_import": "Portable Executable (PE) DLL import event data.",
    "pe_coff:file": "Portable Executable (PE) file event data.",
    "pe_coff:resource": "Portable Executable (PE) resource event data.",
    "plist:key": "Plist event data attribute container.",
    "pls_recall:entry": "PL/SQL Recall event data.",
    "postgresql:application_log:entry": "PostgreSQL application log data.",
    "powershell:transcript_log:entry": "PowerShell transcript log event data.",
    "safari:cookie:entry": "Safari binary cookie event data.",
    "safari:downloads:entry": "Safari download event data.",
    "safari:history:visit": "Safari history event data.",
    "safari:history:visit_sqlite": "Safari history event data.",
    "santa:diskmount": "Santa mount event data.",
    "santa:execution": "Santa execution event data.",
    "santa:file_system_event": "Santa file system event data.",
    "santa:process_exit": "Santa process exit event data.",
    "sccm_log:entry": "SCCM log event data.",
    "selinux:line": "SELinux log event data.",
    "setupapi:log:line": "SetupAPI log event data.",
    "shell:zsh:history": "ZSH history event data.",
    "skydrive:log:entry": "SkyDrive log event data.",
    "skype:event:account": "Skype account event data.",
    "skype:event:call": "Skype call event data.",
    "skype:event:chat": "Skype chat event data.",
    "skype:event:sms": "Skype SMS event data.",
    "skype:event:transferfile": "Skype file transfer event data.",
    "snort:fastlog:alert": "Snort3/Suricata fast-log alert event data.",
    "sophos:av:log": "Sophos anti-virus log event data.",
    "spotlight:metadata_item": "Apple Spotlight store database metadata item event data.",
    "spotlight_searched_terms:entry": "Spotlight searched terms event data.",
    "spotlight_volume_configuration:store": "Spotlight volume configuration event data.",
    "syslog:comment": "Syslog comment event data.",
    "syslog:cron:task_run": "Syslog cron task run event data.",
    "syslog:line": "Syslog line event data.",
    "syslog:ssh:failed_connection": "SSH failed connection event data.",
    "syslog:ssh:login": "SSH login event data.",
    "syslog:ssh:opened_connection": "SSH opened connection event data.",
    "systemd:journal": "Systemd journal event data.",
    "task_scheduler:task_cache:entry": "Task Cache event data.",
    "teamviewer:application_log:entry": "TeamViewer application log event data.",
    "teamviewer:connections_incoming:entry": "TeamViewer incoming connection log event data.",
    "teamviewer:connections_outgoing:entry": "TeamViewer outgoing connection log event data.",
    "viminfo:history": "VimInfo event data.",
    "vsftpd:log": "Vsftpd log event data.",
    "wincc:simatic_s7:entry": "SIMATIC S7 event data.",
    "wincc:sys_log:entry": "WinCC Sys Log event data.",
    "windows:diagnosis:eventtranscript": "Windows diagnosis EventTranscript event data.",
    "windows:distributed_link_tracking:creation": "Windows distributed link event data attribute container.",
    "windows:evt:record": "Windows EventLog (EVT) record event data.",
    "windows:evtx:record": "Windows XML EventLog (EVTX) record event data.",
    "windows:file_history:namespace": "File history namespace table event data.",
    "windows:firewall_log:entry": "Windows Firewall event data.",
    "windows:lnk:link": "Windows Shortcut (LNK) link event data.",
    "windows:metadata:deleted_item": "Windows Recycle Bin event data.",
    "windows:onedrive:log": "OneDrive log event data.",
    "windows:pca_log:entry": "Windows PCA (Program Compatibility Assistant) event data.",
    "windows:prefetch:execution": "Windows Prefetch event data.",
    "windows:registry:amcache": "AMCache file event data.",
    "windows:registry:amcache:programs": "AMCache programs event data.",
    "windows:registry:appcompatcache": "Application Compatibility Cache event data.",
    "windows:registry:bagmru": "BagMRU (or ShellBags) event data attribute container.",
    "windows:registry:bam": "Background Activity Moderator event data.",
    "windows:registry:boot_execute": "Windows Boot Execute event data attribute container.",
    "windows:registry:boot_verification": "Windows Boot Verification event data attribute container.",
    "windows:registry:explorer:programcache": "Explorer ProgramsCache event data attribute container.",
    "windows:registry:installation": "Windows installation event data attribute container.",
    "windows:registry:key_value": "Windows Registry event data attribute container.",
    "windows:registry:motherboard_info": "Windows Motherboard Info event data attribute container.",
    "windows:registry:mount_points2": "Windows MountPoints2 event data attribute container.",
    "windows:registry:mrulist": "MRUList event data attribute container.",
    "windows:registry:mrulistex": "MRUListEx event data attribute container.",
    "windows:registry:msie_zone_settings": "MSIE zone settings event data attribute container.",
    "windows:registry:mstsc:connection": "Terminal Server client connection event data attribute container.",
    "windows:registry:mstsc:mru": "Terminal Server client MRU event data attribute container.",
    "windows:registry:network": "Windows NetworkList event data.",
    "windows:registry:network_drive": "Network drive event data attribute container.",
    "windows:registry:office_mru": "Microsoft Office MRU Windows Registry event data.",
    "windows:registry:office_mru_list": "Microsoft Office MRU list Windows Registry event data.",
    "windows:registry:outlook_search_mru": "Outlook search MRU event data attribute container.",
    "windows:registry:run": "Run/RunOnce key event data attribute container.",
    "windows:registry:sam_users": "Class that defines SAM users Windows Registry event data.",
    "windows:registry:service": "Windows Registry driver or service event data attribute container.",
    "windows:registry:shutdown": "Shutdown Windows Registry event data.",
    "windows:registry:timezone": "Timezone settings event data attribute container.",
    "windows:registry:typedurls": "Typed URLs event data attribute container.",
    "windows:registry:usb": "Windows USB device event data attribute container.",
    "windows:registry:usbstor:instance": "USBStor device instance event data attribute container.",
    "windows:registry:userassist": "UserAssist Windows Registry event data.",
    "windows:registry:winlogon": "Winlogon event data attribute container.",
    "windows:restore_point:info": "Windows Restore Point event data.",
    "windows:shell_item:file_entry": "Windows shell item file entry event data attribute container.",
    "windows:srum:application_usage": "SRUM application resource usage event data.",
    "windows:srum:network_connectivity": "SRUM network connectivity usage event data.",
    "windows:srum:network_usage": "SRUM network data usage event data.",
    "windows:tasks:job": "Windows Scheduled Task event data.",
    "windows:tasks:trigger": "Windows Scheduled Task trigger event data.",
    "windows:timeline:generic": "Windows 10 timeline database generic event data.",
    "windows:timeline:user_engaged": "Windows 10 timeline database User Engaged event data.",
    "windows:user_access_logging:clients": "Windows User Access Logging CLIENTS table event data.",
    "windows:user_access_logging:dns": "Windows User Access Logging DNS table event data.",
    "windows:user_access_logging:role_access": "Windows User Access Logging ROLE_ACCESS table event data.",
    "windows:user_access_logging:system_identity": "Windows User Access Logging SYSTEM_IDENTITY table event data.",
    "windows:user_access_logging:virtual_machines": "Windows User Access Logging VIRTUALMACHINES table event data.",
    "windows:volume:creation": "Windows volume event data attribute container.",
    "windows:wpndatabase:notification": "Windows push notification event data.",
    "windows:wpndatabase:notification_handler": "Windows push notification handler event data.",
    "winrar:history": "WinRAR history event data attribute container.",
    "xchat:log:line": "XChat Log event data.",
    "xchat:scrollback:line": "XChat Scrollback line event data.",
    "zeitgeist:activity": "Zeitgeist activity event data.",
    # These data_types are not present in current Plaso source code but do
    # nonetheless appear in some cases, probably due to old versions of Plaso and
    # change of data_type names.
    "linux:locate": "Linux locate database (updatedb) event data.",
    "bash:history:command": "Bash history log event data.",
    "PLSRecall:event": "PL/SQL Recall event data.",
    "fs:mactime:line": "Mactime filesystem event data.",
    "dpkg:line": "Dpkg event data.",
    "apt:history:line": "APT History log event data.",
    "pe": "Portable Executable (PE).",
}

omit_fields = [
    "data_type",
    "timestamp",
    "timestamp_desc",
    "pathspec",
    "__container_type__",
    "__type__",
    "date_type",
    "date_time",
    "sha256_hash",
    "section_names",
    "parser",
    "display_name",
]


def simple_format(obj) -> str:
    if isinstance(obj, (int, str, float)):
        return str(obj)
    if isinstance(obj, list):
        elements = [simple_format(e) for e in obj]
        return "[" + ", ".join(elements) + "]"
    if isinstance(obj, dict):
        elements = []
        for k, v in obj.items():
            fv = simple_format(v)
            elements.append(f"{k}: {fv}")
        return "{" + ", ".join(elements) + "}"
    if obj is None:
        return "None"
    raise ValueError(f"Unsupported type: {type(obj)}")


def create_db(filename: str):
    with sqlite3.connect(filename) as con:
        cur = con.cursor()
        print("Creating records table")
        cur.execute(
            "CREATE TABLE records (record_id TEXT, log_type TEXT, timestamp_micros INTEGER, timestamp_desc TEXT, message TEXT, enrichment TEXT)"
        )
        print("Creating descriptions table")
        cur.execute("CREATE TABLE log_descriptions (log_type TEXT, description TEXT)")


def populate_records_table(db_file: str, events_file: str):
    with sqlite3.connect(db_file) as con:
        cur = con.cursor()
        next_id = 0
        with open(events_file, "rt") as events:
            for line in events:
                event = json.loads(line)
                log_type = event["data_type"]
                timestamp_desc = event["timestamp_desc"]
                timestamp_micros = event["timestamp"]
                for field in omit_fields:
                    event.pop(field, None)
                message = simple_format(event)
                cur.execute(
                    "INSERT INTO records (record_id, log_type, timestamp_micros, timestamp_desc, message, enrichment) VALUES (?, ?, ?, ?, ?, ?)",
                    (
                        str(next_id),
                        log_type,
                        timestamp_micros,
                        timestamp_desc,
                        message,
                        None,
                    ),
                )
                next_id += 1
                if next_id % 10000 == 0:
                    print(".", end="", flush=True)
        print(f"\nPopulated {next_id} records.")


def populate_descriptions_table(db_file: str, log_type_to_description: dict[str, str]):
    with sqlite3.connect(db_file) as con:
        cur = con.cursor()
        cur.execute("SELECT DISTINCT(log_type) FROM records")
        log_types = cur.fetchall()
        for (log_type,) in log_types:
            desc = log_type_to_description.get(log_type)
            if desc is None:
                print(f"WARNING: missing description for log type {log_type}")
                continue
            cur.execute(
                "INSERT INTO log_descriptions (log_type, description) VALUES (?, ?)",
                (log_type, desc),
            )
        print("Populated log descriptions.")


# Execute helper functions to generate SQLite database
assert os.path.exists("events.jsonl"), (
    "Failed to find 'events.jsonl'. Please ensure the Plaso extraction commands (log2timeline and psort) in the cells above executed successfully."
)

create_db("sauce.db")
populate_records_table("sauce.db", "events.jsonl")
populate_descriptions_table("sauce.db", PLASO_DATA_TYPE_DESCRIPTIONS)